In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8cd319b4f2187b6b29bb69603a96460fc325a975"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8cd319b4f2187b6b29bb69603a96460fc325a975/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Natural Language Inference: Fine-Tuning BERT

In earlier sections of this chapter,
we have designed an attention-based architecture
(in that section)
for the natural language inference task
on the SNLI dataset (as described in that section).
Now we revisit this task by fine-tuning BERT.
As discussed in that section,
natural language inference is a sequence-level text pair classification problem,
and fine-tuning BERT only requires an additional MLP-based architecture,
as illustrated in the figure.

![This section feeds pretrained BERT to an MLP-based architecture for natural language inference.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/nlp-map-nli-bert.svg)

In this section,
we will download a pretrained small version of BERT,
then fine-tune it
for natural language inference on the SNLI dataset.

In [1]:
from d2l import jax as d2l
import jax
from jax import numpy as jnp
from flax import nnx
import optax
import numpy as np
import json
import os

## Loading Pretrained BERT

We have explained how to pretrain BERT on the WikiText-2 dataset in
that section and that section
(note that the original BERT model is pretrained on much bigger corpora).
As discussed in that section,
the original BERT model has hundreds of millions of parameters.
In the following,
we provide two versions of pretrained BERT:
"bert.base" is about as big as the original BERT base model that requires a lot of computational resources to fine-tune,
while "bert.small" is a small version to facilitate demonstration.

In [2]:
d2l.DATA_HUB['bert.base'] = (d2l.DATA_URL + 'bert.base.torch.zip',
                             '225d66f04cae318b841a13d32af3acc165f253ac')
d2l.DATA_HUB['bert.small'] = (d2l.DATA_URL + 'bert.small.torch.zip',
                              'c72329e68a732bef0452e4b96a1c341c8910f81f')

def _load_torch_state_dict(path):
    """Load a PyTorch state-dict file as {str: numpy.ndarray} without torch.

    Works for files saved with ``torch.save(model.state_dict(), path)``
    using pickle protocol 2 (the default for PyTorch <= 2.x CPU tensors).
    """
    import pickle, struct, io
    from collections import OrderedDict

    dtype_info = {
        'FloatStorage': (np.float32, 4), 'HalfStorage': (np.float16, 2),
        'LongStorage': (np.int64, 8),    'IntStorage': (np.int32, 4),
        'DoubleStorage': (np.float64, 8),
    }

    class _StorageRef:
        __slots__ = ('dtype_name', 'key', 'location', 'size')
        def __init__(self, dtype_name, key, location, size):
            self.dtype_name = dtype_name; self.key = key
            self.location = location; self.size = size

    class _TensorRef:
        __slots__ = ('storage', 'offset', 'size', 'stride')
        def __init__(self, storage, offset, size, stride):
            self.storage = storage; self.offset = offset
            self.size = tuple(size); self.stride = tuple(stride)

    class _StorageType:
        __slots__ = ('name',)
        def __init__(self, name): self.name = name
        def __call__(self, _): return self

    class _Unpickler(pickle.Unpickler):
        def find_class(self, module, name):
            if module == 'collections' and name == 'OrderedDict':
                return OrderedDict
            if module == 'torch._utils' and name == '_rebuild_tensor_v2':
                return (lambda s, o, sz, st, *a, **k:
                        _TensorRef(s, o, sz, st))
            if module == 'torch' and name in dtype_info:
                return _StorageType(name)
            return super().find_class(module, name)
        def persistent_load(self, saved_id):
            if isinstance(saved_id, tuple) and saved_id[0] == 'storage':
                st = saved_id[1]
                return _StorageRef(
                    st.name if isinstance(st, _StorageType) else 'FloatStorage',
                    saved_id[2], saved_id[3], saved_id[4])
            raise RuntimeError(f'Unknown persistent id: {saved_id}')

    with open(path, 'rb') as f:
        content = f.read()

    buf = io.BytesIO(content)
    for _ in range(3):                     # skip magic, proto, sysinfo
        _Unpickler(buf).load()
    data = _Unpickler(buf).load()          # OrderedDict of _TensorRef
    storage_keys = _Unpickler(buf).load()  # list of storage keys
    pos = buf.tell()

    key_dtype = {}
    for tref in data.values():
        if isinstance(tref, _TensorRef):
            key_dtype[tref.storage.key] = tref.storage.dtype_name

    storages = {}
    for key in storage_keys:
        dname = key_dtype.get(key, 'FloatStorage')
        np_dt, esz = dtype_info[dname]
        n = struct.unpack('<Q', content[pos:pos+8])[0]; pos += 8
        storages[key] = np.frombuffer(content, np_dt, count=n, offset=pos)
        pos += n * esz

    result = OrderedDict()
    for name, tref in data.items():
        if isinstance(tref, _TensorRef):
            s = storages[tref.storage.key]
            ne = 1
            for d in tref.size:
                ne *= d
            result[name] = s[tref.offset:tref.offset+ne].reshape(
                tref.size).copy()
    return result

Either pretrained BERT model contains a "vocab.json" file that defines the vocabulary set
and a "pretrained.params" file of the pretrained parameters.
We implement the following `load_pretrained_model` function to load pretrained BERT parameters.

In [3]:
def load_pretrained_model(pretrained_model, num_hiddens, ffn_num_hiddens,
                          num_heads, num_blks, dropout, max_len, devices):
    data_dir = d2l.download_extract(pretrained_model)
    # Define an empty vocabulary to load the predefined vocabulary
    vocab = d2l.Vocab()
    vocab.idx_to_token = json.load(open(os.path.join(data_dir, 'vocab.json')))
    vocab.token_to_idx = {token: idx for idx, token in enumerate(
        vocab.idx_to_token)}
    bert = d2l.BERTModel(
        len(vocab), num_hiddens, ffn_num_hiddens=ffn_num_hiddens,
        num_heads=num_heads, num_blks=num_blks, dropout=dropout,
        max_len=max_len)
    # Load pretrained PyTorch BERT parameters (as NumPy) into the NNX graph.
    pt_state_dict = _load_torch_state_dict(
        os.path.join(data_dir, 'pretrained.params'))
    _load_torch_into_nnx_bert(bert, pt_state_dict)
    return bert, vocab

def _load_torch_into_nnx_bert(bert, pt_state_dict):
    """Load a PyTorch BERT state dict into an NNX model."""
    p = pt_state_dict
    # Encoder: token, segment, position embeddings
    enc = bert.encoder
    enc.token_embedding.embedding[...] = jnp.array(
        p['encoder.token_embedding.weight'])
    enc.segment_embedding.embedding[...] = jnp.array(
        p['encoder.segment_embedding.weight'])
    enc.pos_embedding[...] = jnp.array(p['encoder.pos_embedding'])
    # Transformer encoder blocks
    for i, blk in enumerate(enc.blks):
        prefix = f'encoder.blks.{i}'
        # Multi-head attention
        attn = blk.attention
        for name in ['W_q', 'W_k', 'W_v', 'W_o']:
            linear = getattr(attn, name)
            linear.kernel[...] = jnp.array(
                p[f'{prefix}.attention.{name}.weight'].T)
            linear.bias[...] = jnp.array(
                p[f'{prefix}.attention.{name}.bias'])
        # Addnorm layers (LayerNorm)
        for ln_name in ['addnorm1', 'addnorm2']:
            ln = getattr(blk, ln_name).ln
            ln.scale[...] = jnp.array(p[f'{prefix}.{ln_name}.ln.weight'])
            ln.bias[...] = jnp.array(p[f'{prefix}.{ln_name}.ln.bias'])
        # FFN
        ffn = blk.ffn
        ffn.dense1.kernel[...] = jnp.array(
            p[f'{prefix}.ffn.dense1.weight'].T)
        ffn.dense1.bias[...] = jnp.array(p[f'{prefix}.ffn.dense1.bias'])
        ffn.dense2.kernel[...] = jnp.array(
            p[f'{prefix}.ffn.dense2.weight'].T)
        ffn.dense2.bias[...] = jnp.array(p[f'{prefix}.ffn.dense2.bias'])
    # Hidden (tanh) layer
    bert.hidden.kernel[...] = jnp.array(p['hidden.0.weight'].T)
    bert.hidden.bias[...] = jnp.array(p['hidden.0.bias'])

To facilitate demonstration on most machines,
we will load and fine-tune the small version ("bert.small") of the pretrained BERT in this section.
In the exercise, we will show how to fine-tune the much larger "bert.base" to significantly improve the testing accuracy.

In [4]:
devices = d2l.try_all_gpus()
bert, vocab = load_pretrained_model(
    'bert.small', num_hiddens=256, ffn_num_hiddens=512, num_heads=4,
    num_blks=2, dropout=0.1, max_len=512, devices=devices)

## The Dataset for Fine-Tuning BERT

For the downstream task natural language inference on the SNLI dataset,
we define a customized dataset class `SNLIBERTDataset`.
In each example,
the premise and hypothesis form a pair of text sequence
and is packed into one BERT input sequence as depicted in the figure.
Recall that section that segment IDs
are used to distinguish the premise and the hypothesis in a BERT input sequence.
With the predefined maximum length of a BERT input sequence (`max_len`),
the last token of the longer of the input text pair keeps getting removed until
`max_len` is met.
To accelerate generation of the SNLI dataset
for fine-tuning BERT,
we use 4 worker processes to generate training or testing examples in parallel.

In [5]:
class SNLIBERTDataset:
    def __init__(self, dataset, max_len, vocab=None):
        all_premise_hypothesis_tokens = [[
            p_tokens, h_tokens] for p_tokens, h_tokens in zip(
            *[d2l.tokenize([s.lower() for s in sentences])
              for sentences in dataset[:2]])]
        
        self.labels = np.asarray(dataset[2], dtype=np.int32)
        self.vocab = vocab
        self.max_len = max_len
        (self.all_token_ids, self.all_segments,
         self.valid_lens) = self._preprocess(all_premise_hypothesis_tokens)
        print('read ' + str(len(self.all_token_ids)) + ' examples')

    def _preprocess(self, all_premise_hypothesis_tokens):
        # This Python token/list processing is inexpensive enough here that a
        # list comprehension avoids multiprocessing setup and serialization.
        out = [self._preprocess_pair(tokens)
               for tokens in all_premise_hypothesis_tokens]
        all_token_ids = [
            token_ids for token_ids, segments, valid_len in out]
        all_segments = [segments for token_ids, segments, valid_len in out]
        valid_lens = [valid_len for token_ids, segments, valid_len in out]
        return (np.asarray(all_token_ids, dtype=np.int32),
                np.asarray(all_segments, dtype=np.int32),
                np.asarray(valid_lens, dtype=np.float32))

    def _preprocess_pair(self, premise_hypothesis_tokens):
        p_tokens, h_tokens = premise_hypothesis_tokens
        self._truncate_pair_of_tokens(p_tokens, h_tokens)
        tokens, segments = d2l.get_tokens_and_segments(p_tokens, h_tokens)
        token_ids = self.vocab[tokens] + [self.vocab['<pad>']] \
                             * (self.max_len - len(tokens))
        segments = segments + [0] * (self.max_len - len(segments))
        valid_len = len(tokens)
        return token_ids, segments, valid_len

    def _truncate_pair_of_tokens(self, p_tokens, h_tokens):
        # Reserve slots for '<cls>', '<sep>', and '<sep>' tokens for the BERT
        # input
        while len(p_tokens) + len(h_tokens) > self.max_len - 3:
            if len(p_tokens) > len(h_tokens):
                p_tokens.pop()
            else:
                h_tokens.pop()

    def __getitem__(self, idx):
        return (self.all_token_ids[idx], self.all_segments[idx],
                self.valid_lens[idx]), self.labels[idx]

    def __len__(self):
        return len(self.all_token_ids)

After downloading the SNLI dataset,
we generate training and testing examples
by instantiating the `SNLIBERTDataset` class.
Such examples will be read in minibatches during training and testing
of natural language inference.

In [6]:
# Reduce `batch_size` if there is an out of memory error. In the original BERT
# model, `max_len` = 512
batch_size, max_len = 512, 128
data_dir = d2l.download_extract('SNLI')
train_set = SNLIBERTDataset(d2l.read_snli(data_dir, True), max_len, vocab)
test_set = SNLIBERTDataset(d2l.read_snli(data_dir, False), max_len, vocab)
train_iter = d2l.load_array(
    (train_set.all_token_ids, train_set.all_segments,
     train_set.valid_lens, train_set.labels), batch_size, is_train=True)
test_iter = d2l.load_array(
    (test_set.all_token_ids, test_set.all_segments,
     test_set.valid_lens, test_set.labels), batch_size, is_train=False)

read 549367 examples


read 9824 examples


## Fine-Tuning BERT

As the figure indicates,
fine-tuning BERT for natural language inference
requires only an extra MLP consisting of two fully connected layers
(see `self.hidden` and `self.output`---named `self.output_layer` in the
TensorFlow tab to avoid clashing with Keras's reserved `output` property---in
the following `BERTClassifier` class).
This MLP transforms the
BERT representation of the special “&lt;cls&gt;” token,
which encodes the information of both the premise and the hypothesis,
into three outputs of natural language inference:
entailment, contradiction, and neutral.

In [7]:
class BERTClassifier(nnx.Module):
    def __init__(self, bert, rngs=None):
        self.bert = bert
        rngs = nnx.Rngs(0) if rngs is None else rngs
        self.output = nnx.Linear(bert.hidden.out_features, 3, rngs=rngs)

    def __call__(self, tokens_X, segments_X, valid_lens_x):
        encoded_X = self.bert.encoder(tokens_X, segments_X, valid_lens_x)
        return self.output(jnp.tanh(self.bert.hidden(encoded_X[:, 0, :])))

In the following,
the pretrained BERT model `bert` is fed into the `BERTClassifier` instance `net` for
the downstream application.
In common implementations of BERT fine-tuning,
only the parameters of the output layer of the additional MLP (`net.output`, or `net.output_layer` in the TensorFlow tab) will be learned from scratch.
All the parameters of the pretrained BERT encoder (`net.encoder`) and the hidden layer of the additional MLP (`net.hidden`) will be fine-tuned.

In [8]:
net = BERTClassifier(bert)

Recall that
in that section
both the `MaskLM` class and the `NextSentencePred` class
have parameters in their employed MLPs.
These parameters are part of those in the pretrained BERT model
`bert`, and thus part of parameters in `net`.
However, such parameters are only for computing
the masked language modeling loss
and the next sentence prediction loss
during pretraining.
These two loss functions are irrelevant to fine-tuning downstream applications,
thus the parameters of the employed MLPs in 
`MaskLM` and `NextSentencePred` are not updated (and so their gradients become stale) when BERT is fine-tuned.

To allow parameters with stale gradients,
the flag `ignore_stale_grad=True` is set in the `step` function of `d2l.train_batch_ch13`.
We use this function to train and evaluate the model `net` using the training set
(`train_iter`) and the testing set (`test_iter`) of SNLI.
Due to the limited computational resources, the training and testing accuracy
can be further improved: we leave its discussions in the exercises.

In [9]:
lr, num_epochs = 1e-4, 5
optimizer = nnx.Optimizer(net, optax.adam(lr), wrt=nnx.Param)

@nnx.jit
def train_step(net, optimizer, tokens_X, segments_X, valid_lens_x, labels):
    def loss_fn(model):
        logits = model(tokens_X, segments_X, valid_lens_x)
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, labels).mean()
    loss, grads = nnx.value_and_grad(loss_fn)(net)
    optimizer.update(net, grads)
    return loss

@nnx.jit
def eval_step(net, tokens_X, segments_X, valid_lens_x, labels):
    logits = nnx.view(net, deterministic=True)(
        tokens_X, segments_X, valid_lens_x)
    return (logits.argmax(axis=-1) == labels).sum()

for epoch in range(num_epochs):
    train_loss, n_train = jnp.array(0.0), 0
    for batch in train_iter:
        tokens_X, segments_X, valid_lens_x, labels = (
            batch[0], batch[1], batch[2], batch[3])
        loss = train_step(net, optimizer, tokens_X, segments_X,
                          valid_lens_x, labels)
        train_loss += loss * len(labels)
        n_train += len(labels)
    # Evaluate on test set
    n_correct, n_test = jnp.array(0), 0
    for batch in test_iter:
        tokens_X, segments_X, valid_lens_x, labels = (
            batch[0], batch[1], batch[2], batch[3])
        n_correct += eval_step(
            net, tokens_X, segments_X, valid_lens_x, labels)
        n_test += len(labels)
    train_loss, n_correct = float(train_loss), int(n_correct)
    print(f'epoch {epoch + 1}, loss {train_loss / n_train:.4f}, '
          f'test acc {n_correct / n_test:.4f}')

epoch 1, loss 0.7752, test acc 0.7356


epoch 2, loss 0.6329, test acc 0.7565


epoch 3, loss 0.5652, test acc 0.7741


epoch 4, loss 0.5165, test acc 0.7827


epoch 5, loss 0.4798, test acc 0.7856


## Summary

* We can fine-tune the pretrained BERT model for downstream applications, such as natural language inference on the SNLI dataset.
* During fine-tuning, the BERT model becomes part of the model for the downstream application. Parameters that are only related to pretraining loss will not be updated during fine-tuning. 


## Exercises

1. Fine-tune a much larger pretrained BERT model that is about as big as the original BERT base model if your computational resource allows. Set arguments in the `load_pretrained_model` function as: replacing 'bert.small' with 'bert.base', increasing values of `num_hiddens=256`, `ffn_num_hiddens=512`, `num_heads=4`, and `num_blks=2` to 768, 3072, 12, and 12, respectively. By increasing fine-tuning epochs (and possibly tuning other hyperparameters), can you get a testing accuracy higher than 0.86?
1. How to truncate a pair of sequences according to their ratio of length? Compare this pair truncation method and the one used in the `SNLIBERTDataset` class. What are their pros and cons?

[Discussions](https://d2l.discourse.group/t/1526)